# 03 · ACF/PACF corregidas y sensibilidad del LOOKBACK
Correcciones respecto a la celda 13.6 anterior: (1) con N≈10⁵–10⁶ toda correlación es "significativa"; se usa además un umbral de **relevancia práctica** |ρ| ≥ 0.05/0.10; (2) si la ACF no decorrelaciona dentro del máximo calculado, se reporta así y no como un valor; (3) el LOOKBACK se elige **empíricamente en Validación**, sobre los **mismos instantes objetivo** para todas las ventanas, con 3 semillas y criterio de parsimonia.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from statsmodels.tsa.stattools import acf, pacf
from s4.pipeline import preparar_datos, correr_variante
from s4.windows import inicios_comunes
from s4.registry import leer, resumir_semillas
from s4.train import configurar_gpu
from s4 import features as ft
configurar_gpu()
cfg = cargar_config('../config/ablacion_2025_2026.yaml')
datos = preparar_datos(cfg)

In [ ]:
s = datos["train"]["S4"].to_numpy()
MAXLAG = 2 * 1440
a = acf(s, nlags=MAXLAG, fft=True); pa = pacf(s, nlags=400, method="ywm")
ci = 1.96 / np.sqrt(len(s))
print(f"Banda 95 %: ±{ci:.4f}  (con N={len(s):,} casi todo es 'significativo')")
debil = np.where(np.abs(a[1:]) < 0.5)[0]
print("Primer retardo con |ACF| < 0.5:", debil.min() + 1 if len(debil) else f"> {MAXLAG}")
for u in (0.05, 0.10):
    rel = np.where(np.abs(pa[1:]) >= u)[0] + 1
    print(f"PACF |φ| >= {u}: último retardo relevante = {rel.max() if len(rel) else 0} min")
dentro = np.abs(a[1:]) < ci
print("Decorrelación:", "no ocurre dentro de 48 h" if not dentro.any() else f"{np.argmax(dentro)+1} min")

In [ ]:
LOOKBACKS = [30, 60, 120, 180, 240, 360]
H = cfg["ventanas"]["horizon"]
com = {k: inicios_comunes(datos[k].index, LOOKBACKS, H) for k in ("train", "val", "test")}
base = variante(cfg)   # S4 + cíclica + WFL (variante C de la ablación)
for L in LOOKBACKS:
    c = variante(base, **{"ventanas.lookback": L})
    correr_variante(c, datos, ft.CONJUNTOS["B_S4_ciclica"], f"LB{L}", "lookback",
                    starts={k: com[k][L] for k in com})

In [ ]:
reg = leer(ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv", grupo="lookback", particion="val")
tabla = resumir_semillas(reg, por=("modelo",))
tabla.to_csv(ruta(cfg, "reportes") / "tables/tabla_lookback.csv", index=False); tabla

**Regla de decisión (parsimonia):** el LOOKBACK más corto cuyo HSS medio esté a menos de una desviación estándar del mejor. Fijarlo en el YAML (`ventanas.lookback`).